<a href="https://colab.research.google.com/github/simha-ren/EY-Training/blob/main/Prompt_Core_tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from __future__ import annotations

import argparse
import csv
import json
import re
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from langchain_core.prompts import PromptTemplate
from openai import APIStatusError, OpenAI
from google.colab import userdata


# Changed this line to use the current working directory, as __file__ is not defined in notebooks.
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "output"

# Retrieve GEMINI_API_KEY from Colab secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Gemini works with the GPT/OpenAI SDK because Google exposes an
# OpenAI-compatible endpoint. The SDK still sends the standard
# chat.completions request shape, but base_url routes it to Gemini and
# api_key authenticates with a Gemini API key instead of an OpenAI key.
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL_NAME = "gemini-3.5-flash"
MAX_RETRIES_PER_MODEL = 3


EARNINGS_CALL_SNIPPETS = [
    {
        "company": "Northstar Payments",
        "snippet": (
            "Northstar Payments reported Q2 revenue of $248 million, up 18% year over year. "
            "Subscription revenue grew 26%, driven by new merchant onboarding and higher card "
            "processing volume. Operating margin expanded to 21% from 17% last year. Management "
            "raised full-year revenue guidance but noted that enterprise sales cycles are taking "
            "longer in Europe."
        ),
        "reference_summary": (
            "Northstar Payments grew Q2 revenue 18% with stronger subscription revenue and margin "
            "expansion, while raising full-year guidance despite slower European enterprise sales."
        ),
    },
    {
        "company": "Apex Cloud Retail",
        "snippet": (
            "Apex Cloud Retail delivered $412 million in quarterly revenue, a 9% increase from the "
            "prior year. Gross margin declined by 220 basis points due to higher logistics costs and "
            "promotional discounts. The company added 1.2 million active customers, but management "
            "expects near-term pressure on profitability as it invests in same-day delivery."
        ),
        "reference_summary": (
            "Apex Cloud Retail increased revenue 9% and added customers, but margins fell because "
            "of logistics costs, discounts, and delivery investment."
        ),
    },
    {
        "company": "HelioSemis",
        "snippet": (
            "HelioSemis posted Q3 revenue of $690 million, down 4% year over year, as demand from "
            "consumer electronics customers remained weak. Automotive chip revenue rose 31%, partly "
            "offsetting the decline. Inventory days improved from 96 to 74, and management expects "
            "a gradual recovery in the second half if channel demand stabilizes."
        ),
        "reference_summary": (
            "HelioSemis revenue declined 4% on weak consumer electronics demand, though automotive "
            "chips grew strongly and inventory improved ahead of a possible second-half recovery."
        ),
    },
]


TICKETS = [
    {
        "ticket_id": "T001",
        "text": "My card was charged twice for the same invoice. Please correct the bill.",
        "ground_truth": "Billing",
    },
    {
        "ticket_id": "T002",
        "text": "The dashboard returns a 500 error whenever I open the integrations page.",
        "ground_truth": "Tech",
    },
    {
        "ticket_id": "T003",
        "text": "I cancelled within the trial period and need my payment refunded.",
        "ground_truth": "Refund",
    },
    {
        "ticket_id": "T004",
        "text": "Can you tell me where to download last month's usage statement?",
        "ground_truth": "General",
    },
    {
        "ticket_id": "T005",
        "text": "Our production account is locked and payroll processing is blocked today.",
        "ground_truth": "Escalate",
    },
]


@dataclass
class PromptLogRecord:
    task: str
    prompt_name: str
    variables: dict[str, Any]
    prompt: str
    response: str


class PromptLayerLogger:
    """Small local PromptLayer-style logger for classroom/offline runs."""

    def __init__(self, output_path: Path) -> None:
        self.output_path = output_path
        self.output_path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, record: PromptLogRecord) -> None:
        payload = {
            "timestamp": datetime.now(UTC).isoformat(timespec="seconds"),
            "task": record.task,
            "prompt_name": record.prompt_name,
            "variables": record.variables,
            "prompt": record.prompt,
            "response": record.response,
        }
        with self.output_path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(payload) + "\n")


def create_client() -> OpenAI:
    return OpenAI(api_key=GEMINI_API_KEY, base_url=GEMINI_BASE_URL)


def call_gemini(prompt: str, mock: bool = False) -> str:
    if mock:
        return mock_response(prompt)

    client = create_client()
    messages = [
        {
            "role": "system",
            "content": "You are a precise FinTech NLP assistant. Return concise, useful answers.",
        },
        {"role": "user", "content": prompt},
    ]

    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.2,
            )
            return response.choices[0].message.content or ""
        except APIStatusError as exc:
            last_error = exc
            if exc.status_code not in {429, 500, 502, 503, 504}:
                raise
            wait_seconds = 2 ** (attempt - 1)
            print(
                f"Model {MODEL_NAME} returned {exc.status_code}. "
                f"Retry {attempt}/{MAX_RETRIES_PER_MODEL} after {wait_seconds}s..."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(
        "Gemini API stayed unavailable after retries. "
        "Try again later or run with --mock for offline output generation."
    ) from last_error


def create_zero_shot_summary_prompt() -> PromptTemplate:
    return PromptTemplate.from_template(
        """
Summarise the earnings call snippet for an investor.

Rules:
- Use 2 concise bullet points.
- Mention revenue movement, margin/profitability, guidance, or risks if present.
- Do not invent facts.

Company: {company}
Snippet:
{snippet}

Summary:
""".strip()
    )


def create_few_shot_summary_prompt() -> PromptTemplate:
    return PromptTemplate.from_template(
        """
You summarise earnings call snippets into investor-ready bullets.

Example 1:
Company: FinEdge Analytics
Snippet: FinEdge grew revenue 14% to $120 million. Net retention improved to 118%, but
management lowered operating margin guidance because of hiring in customer success.
Summary:
- Revenue grew 14% to $120 million, supported by stronger customer retention.
- Margin guidance was lowered because the company is investing in customer success.

Example 2:
Company: Orbit Logistics
Snippet: Orbit revenue fell 6% as freight demand weakened. Fuel costs improved and free
cash flow turned positive. Management expects volumes to recover slowly next year.
Summary:
- Revenue declined 6% because freight demand remained weak.
- Lower fuel costs helped cash flow, but management expects only a slow volume recovery.

Now summarise this snippet.

Company: {company}
Snippet:
{snippet}

Summary:
""".strip()
    )


def create_ticket_classifier_prompt() -> PromptTemplate:
    return PromptTemplate.from_template(
        """
Classify the support ticket into exactly one category:
Billing, Tech, Refund, General, Escalate.

Use brief step-by-step reasoning internally, but only output valid JSON:
{{
  "category": "Billing|Tech|Refund|General|Escalate",
  "reason": "one short reason"
}}

Category guide:
- Billing: invoices, charges, pricing, payment issues.
- Tech: bugs, errors, broken features, login or integration failures.
- Refund: cancellation refund, repayment, money-back request.
- General: how-to questions or ordinary information requests.
- Escalate: urgent business impact, production outage, legal/compliance, blocked critical workflow.

Ticket:
{ticket}
""".strip()
    )


def run_summarisation_chain(mock: bool, logger: PromptLayerLogger) -> list[dict[str, Any]]:
    results = []
    zero_shot_prompt = create_zero_shot_summary_prompt()
    few_shot_prompt = create_few_shot_summary_prompt()

    for item in EARNINGS_CALL_SNIPPETS:
        zero_prompt = zero_shot_prompt.format(**item)
        zero_summary = call_gemini(zero_prompt, mock=mock)
        logger.log(
            PromptLogRecord(
                task="summarisation",
                prompt_name="zero_shot_earnings_summary",
                variables={"company": item["company"], "snippet": item["snippet"]},
                prompt=zero_prompt,
                response=zero_summary,
            )
        )

        few_prompt = few_shot_prompt.format(**item)
        few_summary = call_gemini(few_prompt, mock=mock)
        rouge_l = rouge_l_score(few_summary, item["reference_summary"])
        logger.log(
            PromptLogRecord(
                task="summarisation",
                prompt_name="few_shot_earnings_summary",
                variables={"company": item["company"], "snippet": item["snippet"]},
                prompt=few_prompt,
                response=few_summary,
            )
        )

        results.append(
            {
                "company": item["company"],
                "zero_shot_summary": zero_summary,
                "few_shot_summary": few_summary,
                "reference_summary": item["reference_summary"],
                "rouge_l_f1": rouge_l["f1"],
                "rouge_l_precision": rouge_l["precision"],
                "rouge_l_recall": rouge_l["recall"],
            }
        )
    return results


def run_ticket_classifier(mock: bool, logger: PromptLayerLogger) -> list[dict[str, Any]]:
    prompt_template = create_ticket_classifier_prompt()
    results = []

    for ticket in TICKETS:
        prompt = prompt_template.format(ticket=ticket["text"])
        response = call_gemini(prompt, mock=mock)
        logger.log(
            PromptLogRecord(
                task="classification",
                prompt_name="five_class_ticket_classifier",
                variables={"ticket_id": ticket["ticket_id"], "ticket": ticket["text"]},
                prompt=prompt,
                response=response,
            )
        )

        category, reason = parse_classifier_response(response)
        results.append(
            {
                "ticket_id": ticket["ticket_id"],
                "ticket": ticket["text"],
                "ground_truth": ticket["ground_truth"],
                "predicted_category": category,
                "reason": reason,
                "is_correct": category == ticket["ground_truth"],
            }
        )
    return results


def parse_classifier_response(response: str) -> tuple[str, str]:
    try:
        parsed = json.loads(response)
        return str(parsed.get("category", "")), str(parsed.get("reason", ""))
    except json.JSONDecodeError:
        category_match = re.search(
            r"\b(Billing|Tech|Refund|General|Escalate)\b", response, flags=re.IGNORECASE
        )
        category = category_match.group(1).title() if category_match else "Unknown"
        return category, response.strip()


def rouge_l_score(candidate: str, reference: str) -> dict[str, float]:
    candidate_tokens = tokenize(candidate)
    reference_tokens = tokenize(reference)
    if not candidate_tokens or not reference_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    lcs = longest_common_subsequence_length(candidate_tokens, reference_tokens)
    precision = lcs / len(candidate_tokens)
    recall = lcs / len(reference_tokens)
    f1 = 0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall)
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def longest_common_subsequence_length(left: list[str], right: list[str]) -> int:
    previous = [0] * (len(right) + 1)
    for left_token in left:
        current = [0]
        for index, right_token in enumerate(right, start=1):
            if left_token == right_token:
                current.append(previous[index - 1] + 1)
            else:
                current.append(max(previous[index], current[-1]))
        previous = current
    return previous[-1]


def mock_response(prompt: str) -> str:
    if "Classify the support ticket" in prompt:
        ticket_match = re.search(r"Ticket:\s*(.+)", prompt, flags=re.DOTALL)
        ticket_text = ticket_match.group(1) if ticket_match else prompt
        lowered = ticket_text.lower()
        if "charged" in lowered or "invoice" in lowered:
            return json.dumps({"category": "Billing", "reason": "The ticket is about charges."})
        if "500 error" in lowered or "dashboard" in lowered:
            return json.dumps({"category": "Tech", "reason": "The ticket reports a software error."})
        if "refund" in lowered or "cancelled" in lowered:
            return json.dumps({"category": "Refund", "reason": "The user is requesting money back."})
        if "production" in lowered or "blocked" in lowered:
            return json.dumps({"category": "Escalate", "reason": "The issue blocks a critical workflow."})
        return json.dumps({"category": "General", "reason": "The ticket is an information request."})

    company_matches = re.findall(r"Company:\s*(.+)", prompt)
    company = company_matches[-1].strip() if company_matches else "The company"
    return (
        f"- {company} reported material movement in revenue and operating metrics.\n"
        "- Management highlighted key risks and forward-looking expectations."
    )


def save_summarisation_results(rows: list[dict[str, Any]]) -> None:
    path = OUTPUT_DIR / "summarisation_results.csv"
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def save_classifier_results(rows: list[dict[str, Any]]) -> None:
    path = OUTPUT_DIR / "ticket_classifier_results.csv"
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def save_report(summary_rows: list[dict[str, Any]], classifier_rows: list[dict[str, Any]]) -> None:
    average_rouge_l = sum(row["rouge_l_f1"] for row in summary_rows) / len(summary_rows)
    accuracy = sum(row["is_correct"] for row in classifier_rows) / len(classifier_rows)
    report = {
        "core_tasks_completed": [
            "1. Zero-shot + few-shot earnings-call summarisation using LangChain PromptTemplate",
            "3. Five-class ticket classifier with brief reasoning",
            "4. Prompt logging and ROUGE-L scoring for summarisation",
        ],
        "model": MODEL_NAME,
        "sdk": "OpenAI Python SDK",
        "provider": "Gemini via OpenAI-compatible endpoint",
        "average_rouge_l_f1": round(average_rouge_l, 4),
        "ticket_classifier_accuracy": round(accuracy, 4),
        "outputs": [
            "summarisation_results.csv",
            "ticket_classifier_results.csv",
            "promptlayer_log.jsonl",
            "task3_report.json",
        ],
    }
    (OUTPUT_DIR / "task3_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Day 11 Task 3: Prompt Studio core tasks.")
    parser.add_argument(
        "--mock",
        action="store_true",
        help="Run without calling Gemini. Useful for testing output files offline.",
    )
    # In interactive environments like Colab/Jupyter, sys.argv often contains
    # arguments passed by the kernel (e.g., '-f'). To prevent argparse from
    # attempting to parse these and exiting, we explicitly pass an empty list
    # of arguments to parse when running in this context.
    args = parser.parse_args(['--mock'])
    return args


def main() -> None:
    args = parse_args()
    OUTPUT_DIR.mkdir(exist_ok=True)
    prompt_log_path = OUTPUT_DIR / "promptlayer_log.jsonl"
    if prompt_log_path.exists():
        prompt_log_path.unlink()

    logger = PromptLayerLogger(prompt_log_path)
    summary_rows = run_summarisation_chain(mock=args.mock, logger=logger)
    classifier_rows = run_ticket_classifier(mock=args.mock, logger=logger)

    save_summarisation_results(summary_rows)
    save_classifier_results(classifier_rows)
    save_report(summary_rows, classifier_rows)

    print("Task 3 complete.")
    print(f"Mode: {'mock/offline' if args.mock else 'Gemini API'}")
    print(f"Output folder: {OUTPUT_DIR}")
    print("Completed core tasks: 1, 3, 4")


if __name__ == "__main__":
    main()

Task 3 complete.
Mode: mock/offline
Output folder: /content/output
Completed core tasks: 1, 3, 4
